# ReSnap v5, langkah 1: analisis kesalahan model v4b

Tujuan: mencari tahu **kenapa** kelas lemah gagal sebelum mengubah apa pun, supaya v5 memperbaiki penyebab yang benar.

Aturan evaluasi v5 (ditetapkan sebelum training):
- Semua analisis dan pemilihan model hanya memakai **val (1.069 foto)**.
- **Test lama (654 foto) tidak dibuka** di notebook ini.
- Label val yang tampak salah **dicatat, tidak diedit**. Perbaikan label hanya untuk train.

Kebutuhan Kaggle: Accelerator **GPU T4**, **Internet on**, Input: `dataset_v4.zip` (dataset yang sama dengan run v4b).
Model v4b diambil dari GitHub `AffanAbya30/resnap-cv` dan dicek sha256-nya.

Keluaran: `/kaggle/working/analisis_v4b.zip` (kecil, kirim ke chat).

In [ ]:
%pip -q install ultralytics==8.3.228

In [ ]:
import json, math, shutil, subprocess, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps

assert torch.cuda.is_available(), 'GPU tidak aktif. Settings > Accelerator > GPU T4.'
print('GPU:', torch.cuda.get_device_name(0))

INPUT, OUTPUT, TMP = Path('/kaggle/input'), Path('/kaggle/working'), Path('/tmp/resnap')
OUT = OUTPUT / 'analisis_v4b'
OUT.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

REPO = TMP / 'resnap-cv'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AffanAbya30/resnap-cv.git', str(REPO)], check=True)
import hashlib
V4B_SHA = '1410f11d9c2cdfad074446d1742ed5ec24d0269122bc51ec5ddde69bafc66809'
WEIGHTS = REPO / 'models/best.pt'
assert hashlib.sha256(WEIGHTS.read_bytes()).hexdigest() == V4B_SHA, 'models/best.pt bukan v4b'
print('Model v4b OK')

def is_root(p):
    return (p / 'val/images').is_dir() and (p / 'train/images').is_dir() and (p / 'holdout_oi/images').is_dir()

def find_data():
    for p in sorted(INPUT.rglob('*'), key=lambda p: len(p.parts)):
        if p.is_dir() and is_root(p):
            return p
    z = next((p for p in INPUT.rglob('dataset_v4.zip')), None)
    assert z, 'dataset_v4.zip tidak ada di Input'
    dest = TMP / 'data'
    with zipfile.ZipFile(z) as f:
        f.extractall(dest)
    return next(p for p in sorted([dest, *dest.rglob('*')], key=lambda p: len(p.parts)) if p.is_dir() and is_root(p))

DATA = find_data()
VAL_IMG, VAL_LBL = DATA / 'val/images', DATA / 'val/labels'
TRAIN_IMG, TRAIN_LBL = DATA / 'train/images', DATA / 'train/labels'
val_imgs = sorted(p for p in VAL_IMG.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'})
print('Data:', DATA, '| val:', len(val_imgs))
assert len(val_imgs) == 1069, 'Val harus 1.069 foto (sama dengan v3/v4/v4b)'

from ultralytics import YOLO
model = YOLO(str(WEIGHTS))
NAMES = model.names
NC = len(NAMES)
import yaml
yaml_val = TMP / 'val.yaml'
yaml_val.write_text(yaml.safe_dump({'path': str(DATA), 'train': 'val/images', 'val': 'val/images',
                                    'names': {int(k): v for k, v in NAMES.items()}}))

## 2. Metrik val per kelas dan confusion matrix (angka mentah)

In [ ]:
m = model.val(data=str(yaml_val), split='val', imgsz=640, batch=16, conf=0.001, iou=0.7,
              plots=True, project=str(TMP / 'val'), name='v4b', exist_ok=True, device='0', verbose=False)
box = m.box
per_cls = pd.DataFrame({
    'kelas': [NAMES[int(c)] for c in box.ap_class_index],
    'P': box.p, 'R': box.r, 'AP50': box.ap50, 'AP50_95': box.ap,
}).set_index('kelas')
print(f"Val v4b: P {box.mp:.3f}  R {box.mr:.3f}  mAP50 {box.map50:.3f}  mAP50-95 {box.map:.3f}")
display(per_cls.sort_values('AP50').head(10).style.format(precision=3).set_caption('10 kelas terlemah di val'))

# Matrix Ultralytics: baris = prediksi, kolom = label asli, indeks terakhir = background
cm = m.confusion_matrix.matrix.astype(int)
labels = [NAMES[i] for i in range(NC)] + ['background']
pairs = [(labels[j], labels[i], cm[i, j]) for i in range(NC) for j in range(NC) if i != j and cm[i, j] > 0]
pairs = pd.DataFrame(pairs, columns=['asli', 'diprediksi', 'jumlah']).sort_values('jumlah', ascending=False)
display(pairs.head(20).style.set_caption('20 pasangan kelas paling sering tertukar (conf 0.25, IoU 0.45)'))
fn_bg = pd.Series(cm[NC, :NC], index=labels[:NC], name='terlewat (FN)')
fp_bg = pd.Series(cm[:NC, NC], index=labels[:NC], name='deteksi palsu (FP)')
pairs.to_csv(OUT / 'pasangan_tertukar.csv', index=False)
pd.DataFrame(cm, index=[f'pred_{l}' for l in labels], columns=[f'asli_{l}' for l in labels]).to_csv(OUT / 'confusion_matrix_mentah.csv')
for f in ('confusion_matrix_normalized.png',):
    shutil.copy2(TMP / 'val/v4b' / f, OUT / f)

## 3. Pencocokan per objek: terlewat, tertukar, palsu, dan pengaruh ukuran objek

In [ ]:
def read_gt(img_path, w, h):
    lbl = VAL_LBL / f'{img_path.stem}.txt'
    out = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            v = line.split()
            if len(v) < 5:
                continue
            c, cx, cy, bw, bh = int(v[0]), *map(float, v[1:5])
            out.append((c, (cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h))
    return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def size_bucket(b, w, h):
    r = (b[2] - b[0]) * (b[3] - b[1]) / (w * h)
    return 'kecil (<1%)' if r < 0.01 else ('sedang (1-10%)' if r < 0.10 else 'besar (>10%)')

records = []   # satu baris per objek asli dan per deteksi palsu
CONF = 0.25
for i in range(0, len(val_imgs), 32):
    chunk = val_imgs[i:i + 32]
    for res in model.predict([str(p) for p in chunk], conf=CONF, iou=0.7, imgsz=640, device='0', verbose=False, stream=True):
        path = Path(res.path); h, w = res.orig_shape
        gts = read_gt(path, w, h)
        preds = [(int(c), tuple(map(float, b)), float(s)) for b, c, s in
                 zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy())]
        preds.sort(key=lambda x: -x[2])
        used_p, used_g = set(), set()
        # 1) benar: kelas sama, IoU >= 0.5
        for pi, (pc, pb, ps) in enumerate(preds):
            best, bj = 0.5, None
            for gj, (gc, *gb) in enumerate(gts):
                if gj in used_g or gc != pc:
                    continue
                v = iou(pb, gb)
                if v >= best:
                    best, bj = v, gj
            if bj is not None:
                used_p.add(pi); used_g.add(bj)
                records.append(dict(img=path.name, asli=NAMES[gts[bj][0]], pred=NAMES[pc], status='benar', conf=ps,
                                    ukuran=size_bucket(gts[bj][1:], w, h), box=gts[bj][1:]))
        # 2) tertukar: kelas beda, IoU >= 0.5
        for pi, (pc, pb, ps) in enumerate(preds):
            if pi in used_p:
                continue
            best, bj = 0.5, None
            for gj, (gc, *gb) in enumerate(gts):
                if gj in used_g:
                    continue
                v = iou(pb, gb)
                if v >= best:
                    best, bj = v, gj
            if bj is not None:
                used_p.add(pi); used_g.add(bj)
                records.append(dict(img=path.name, asli=NAMES[gts[bj][0]], pred=NAMES[pc], status='tertukar', conf=ps,
                                    ukuran=size_bucket(gts[bj][1:], w, h), box=gts[bj][1:]))
        # 3) terlewat dan palsu
        for gj, (gc, *gb) in enumerate(gts):
            if gj not in used_g:
                records.append(dict(img=path.name, asli=NAMES[gc], pred=None, status='terlewat', conf=None,
                                    ukuran=size_bucket(gb, w, h), box=tuple(gb)))
        for pi, (pc, pb, ps) in enumerate(preds):
            if pi not in used_p:
                records.append(dict(img=path.name, asli=None, pred=NAMES[pc], status='palsu', conf=ps,
                                    ukuran=size_bucket(pb, w, h), box=pb))

rec = pd.DataFrame(records)
gt = rec[rec.asli.notna()]
ringkas = gt.pivot_table(index='asli', columns='status', values='img', aggfunc='count', fill_value=0)
ringkas['total'] = ringkas.sum(axis=1)
ringkas['recall@0.25'] = ringkas.get('benar', 0) / ringkas['total']
ringkas['palsu'] = rec[rec.status == 'palsu'].groupby('pred').size().reindex(ringkas.index, fill_value=0)
ringkas = ringkas.join(per_cls[['AP50']]).sort_values('AP50')
display(ringkas.head(12).style.format({'recall@0.25': '{:.2f}', 'AP50': '{:.3f}'}).set_caption('Nasib setiap objek asli, kelas terlemah dulu'))

uk = gt.assign(ok=gt.status == 'benar').pivot_table(index='asli', columns='ukuran', values='ok', aggfunc='mean')
uk_n = gt.pivot_table(index='asli', columns='ukuran', values='img', aggfunc='count', fill_value=0)
display(uk.loc[ringkas.index[:12]].style.format('{:.2f}', na_rep='-').set_caption('Recall per ukuran objek (kelas terlemah)'))
print('Recall keseluruhan per ukuran:', gt.assign(ok=gt.status == 'benar').groupby('ukuran').ok.mean().round(3).to_dict())
rec.drop(columns='box').to_csv(OUT / 'per_objek_val.csv', index=False)
ringkas.to_csv(OUT / 'ringkasan_per_kelas.csv')
uk.join(uk_n, rsuffix='_n').to_csv(OUT / 'recall_per_ukuran.csv')

## 4. Kualitas anotasi: kotak tumpukan dan kotak raksasa (train dan val)

In [ ]:
def scan_labels(lbl_dir, img_dir):
    rows = []
    for lbl in lbl_dir.glob('*.txt'):
        if '_dup' in lbl.stem:
            continue
        boxes = []
        for line in lbl.read_text().splitlines():
            v = line.split()
            if len(v) >= 5:
                c, cx, cy, bw, bh = int(v[0]), *map(float, v[1:5])
                boxes.append((c, cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2, bw * bh))
        for k, (c, x1, y1, x2, y2, a) in enumerate(boxes):
            inside = 0
            for j, (c2, a1, b1, a2, b2, aa) in enumerate(boxes):
                if j == k or c2 != c or aa >= a:
                    continue
                ix = max(0, min(x2, a2) - max(x1, a1)); iy = max(0, min(y2, b2) - max(y1, b1))
                if aa > 0 and ix * iy / aa >= 0.8:
                    inside += 1
            rows.append(dict(file=lbl.stem, kelas=NAMES[c], luas=a, n_kotak_foto=len(boxes),
                             tumpukan=inside >= 2, raksasa=a >= 0.5))
    return pd.DataFrame(rows)

ann = {s: scan_labels(DATA / s / 'labels', DATA / s / 'images') for s in ('train', 'val')}
tab = pd.concat({s: d.groupby('kelas').agg(kotak=('file', 'size'), foto=('file', 'nunique'),
                                           tumpukan=('tumpukan', 'sum'), raksasa=('raksasa', 'sum'))
                 for s, d in ann.items()}, axis=1)
tab.columns = [f'{a}_{b}' for a, b in tab.columns]
tab = tab.join(per_cls[['AP50']]).sort_values('AP50')
display(tab.head(12).style.format({'AP50': '{:.3f}'}).set_caption('Jumlah data dan kotak bermasalah per kelas'))
tab.to_csv(OUT / 'data_dan_anotasi_per_kelas.csv')
print('Kotak "tumpukan" = kotak yang membungkus >= 2 kotak lain berkelas sama (anotasi ganda: per tumpukan dan per objek).')
print('Kotak "raksasa" = kotak >= 50% luas foto.')

## 5. Lembar contoh visual untuk kelas terlemah

In [ ]:
WEAK = list(dict.fromkeys(list(ringkas.index[:6]) + ['turmeric', 'galangal', 'shrimp', 'broccoli', 'mushroom', 'garlic']))
WEAK = [k for k in WEAK if k in ringkas.index]
TILE = 150

def crop(img_name, b, label):
    im = ImageOps.exif_transpose(Image.open(VAL_IMG / img_name)).convert('RGB')
    w, h = im.size
    pw, ph = (b[2] - b[0]) * 0.15, (b[3] - b[1]) * 0.15
    c = im.crop((max(0, b[0] - pw), max(0, b[1] - ph), min(w, b[2] + pw), min(h, b[3] + ph)))
    c.thumbnail((TILE, TILE - 18))
    tile = Image.new('RGB', (TILE, TILE), 'white')
    tile.paste(c, ((TILE - c.width) // 2, 18 + (TILE - 18 - c.height) // 2))
    ImageDraw.Draw(tile).text((3, 2), label[:24], fill='black')
    return tile

def sheet(rows, title, path, cols=8):
    if rows.empty:
        return
    tiles = [crop(r.img, r.box, r.label) for r in rows.itertuples()]
    n = len(tiles); rws = math.ceil(n / cols)
    s = Image.new('RGB', (cols * TILE, rws * TILE + 22), (245, 245, 245))
    ImageDraw.Draw(s).text((4, 4), title, fill='black')
    for k, t in enumerate(tiles):
        s.paste(t, ((k % cols) * TILE, 22 + (k // cols) * TILE))
    s.save(path, quality=80)

for k in WEAK:
    g = gt[gt.asli == k]
    tert = g[g.status == 'tertukar'].assign(label=lambda d: 'jadi ' + d.pred.astype(str))
    lewat = g[g.status == 'terlewat'].assign(label=lambda d: 'terlewat ' + d.ukuran.str.split().str[0])
    palsu = rec[(rec.status == 'palsu') & (rec.pred == k)].sort_values('conf', ascending=False).assign(
        label=lambda d: 'palsu ' + d.conf.round(2).astype(str))
    rows = pd.concat([tert.head(12), lewat.sample(min(12, len(lewat)), random_state=0), palsu.head(8)])
    sheet(rows, f'{k}: tertukar / terlewat / deteksi palsu (val, conf 0.25)', OUT / f'contoh_{k}.jpg')
print('Lembar contoh:', sorted(p.name for p in OUT.glob('contoh_*.jpg')))

## 6. Cek sumber data tambahan (tanpa unduh)

Daftar kandidat dari Roboflow Universe untuk kunyit dan lengkuas (Open Images tidak punya kelas ini). Sel ini hanya membaca metadata (tipe proyek, jumlah foto, kelas, lisensi) lewat API Roboflow; belum ada yang diunduh. Butuh API key Roboflow yang sama dengan saat audit dataset awal.

In [ ]:
import requests
from getpass import getpass
CANDIDATES = [
    'research-uc/rempah-rempah', 'soda-fj5ov/deteksi-rempah-rempah-2', 'fruits-2kbl9/detection-ginger-t9kqu',
    'rimpang-key8c/rempah-c9tpp', 'pcd-5j06s/rimpang-0gbw2-oszc5', 'sesilia/210218748_sesilia_bumbu-dapur',
    'aidetection-e29es/hueheueeh', 'rival-0zxtv/rempah-r4adj', 'relz-f3q8v/deteksi-rempah-jjceg',
    'data-66vnh/turmeric', 'hritik-lwytm/turmeric_data', 'ayushvision/ayushvision_tuber',
    'lipika/ingredient-detection-skigf', 'cale/membedakan-tanaman-rempah',
]
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
except Exception:
    key = None
    print('Secret ROBOFLOW_API_KEY belum ada: cek kandidat Roboflow dilewati.')
meta = []
for slug in (CANDIDATES if key else []):
    try:
        r = requests.get(f'https://api.roboflow.com/{slug}', params={'api_key': key}, timeout=30)
        j = r.json()
        p = j.get('project', {})
        vers = j.get('versions', [])
        last = max(vers, key=lambda v: v.get('created', 0)) if vers else {}
        meta.append(dict(proyek=slug, tipe=p.get('type'), foto=p.get('images'), lisensi=p.get('license'),
                         kelas=', '.join(sorted((p.get('classes') or {}).keys()))[:120],
                         versi_terakhir=str(last.get('id', '')).split('/')[-1], foto_versi=last.get('images'),
                         status=r.status_code))
    except Exception as e:
        meta.append(dict(proyek=slug, status=f'error: {e}'))
del key
meta = pd.DataFrame(meta)
if len(meta):
    display(meta)
meta.to_csv(OUT / 'kandidat_roboflow.csv', index=False)

# Kelas Open Images (untuk brokoli, jamur, bawang putih, udang, dll.)
oi = pd.read_csv('https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv', header=None, names=['id', 'nama'])
cek = ['Broccoli', 'Mushroom', 'Garlic', 'Shrimp', 'Ginger', 'Potato', 'Cabbage', 'Carrot', 'Tomato', 'Cucumber',
       'Egg (Food)', 'Lemon', 'Pumpkin', 'Zucchini', 'Bell pepper']
ada = oi[oi.nama.isin(cek)]
print('Kelas Open Images yang tersedia:', sorted(ada.nama))
print('Tidak ada di Open Images:', sorted(set(cek) - set(ada.nama)))

## 7. Paket hasil

In [ ]:
summary = {
    'val_v4b': {'P': float(box.mp), 'R': float(box.mr), 'mAP50': float(box.map50), 'mAP50_95': float(box.map)},
    'kelas_terlemah_AP50': per_cls.AP50.sort_values().head(8).round(3).to_dict(),
    'pasangan_tertukar_top10': pairs.head(10).values.tolist(),
    'recall_per_ukuran': gt.assign(ok=gt.status == 'benar').groupby('ukuran').ok.mean().round(3).to_dict(),
    'kotak_tumpukan_train': int(ann['train'].tumpukan.sum()), 'kotak_raksasa_train': int(ann['train'].raksasa.sum()),
}
(OUT / 'ringkasan.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
zp = OUTPUT / 'analisis_v4b.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        z.write(f, f.name)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Siap dikirim:', zp, round(zp.stat().st_size / 1e6, 2), 'MB')